# 01. 同步与并发基础

这份教程面向有 Java 经验的 Python 初学者。学完后，你应该能够：

- 区分同步/异步、阻塞/非阻塞、并发/并行；
- 判断任务属于 I/O 密集型还是 CPU 密集型；
- 理解默认 CPython 中 GIL 对多线程的影响；
- 理解 Python 与 Java 常见并发抽象之间的大致对应关系；
- 根据任务特征选择线程、进程或异步。

本章只使用标准库，所有“等待”都由短暂的 `sleep` 离线模拟。

## 1. 先从同步开始

**同步（synchronous）**描述调用关系：调用者发起操作后，要等操作完成并取得结果，才能继续下一步。

```text
主流程：任务 A ──────────> 任务 B ──────────> 任务 C
时间：  0s             0.2s             0.4s
```

它很像 Java 中连续调用三个普通方法：代码顺序就是执行顺序，最容易阅读和调试。

In [3]:
from time import perf_counter, sleep


def fetch_sync(name: str, delay: float = 0.2) -> str:
    # 用 sleep 模拟等待网络或磁盘。
    print(f"{name} 开始")
    sleep(delay)
    print(f"{name} 完成")
    return name


started = perf_counter()
sync_results = [fetch_sync(f"任务 {index}") for index in range(3)]
sync_elapsed = perf_counter() - started

print("结果：", sync_results)
print(f"总耗时约 {sync_elapsed:.2f} 秒")

任务 0 开始
任务 0 完成
任务 1 开始
任务 1 完成
任务 2 开始
任务 2 完成
结果： ['任务 0', '任务 1', '任务 2']
总耗时约 0.61 秒


## 2. 同步/异步与阻塞/非阻塞不是同一组概念

| 维度 | 关注的问题 | 含义 |
|---|---|---|
| 同步 / 异步 | **结果如何交付？** | 同步调用直接等待结果；异步调用允许稍后取得结果 |
| 阻塞 / 非阻塞 | **等待时线程能否继续？** | 阻塞会停住当前线程；非阻塞会立即返回或让出执行权 |

初学阶段可以先这样记：

- `time.sleep()`：同步且阻塞当前线程；
- `await asyncio.sleep()`：当前协程暂停，但事件循环线程可以运行其他协程；
- `Future`/`Task`：代表“将来会得到的结果”，类似 Java 的 `Future`/`CompletableFuture`，但调度模型并不完全相同。

异步并不自动等于并行，也不保证更快。只有任务中存在可以重叠的等待时，异步才通常有明显收益。

## 3. 并发与并行

- **并发（concurrency）**：多个任务在同一段时间内都有进展，重点是组织和调度任务。
- **并行（parallelism）**：多个任务在同一时刻真正执行，通常需要多个 CPU 核心或其他计算单元。

```text
并发（单个执行资源交替工作）
CPU：A A | B B | A | B

并行（两个执行资源同时工作）
CPU1：A A A A
CPU2：B B B B
```

`asyncio` 默认在一个线程中并发调度协程；多进程则可以让纯 Python 计算在多个 CPU 核心上并行。

## 4. I/O 密集型与 CPU 密集型

| 类型 | 时间主要花在哪里 | 例子 | 常见选择 |
|---|---|---|---|
| I/O 密集型 | 等网络、磁盘、数据库、用户输入 | HTTP 请求、数据库查询 | `asyncio`、AnyIO、线程池 |
| CPU 密集型 | 执行大量计算 | 图像处理、压缩、纯 Python 数值循环 | 多进程、原生扩展、分布式计算 |

线程或异步能把 I/O 等待时间重叠起来；它们不会凭空减少 CPU 计算量。

In [4]:
def classify_task(wait_ratio: float) -> str:
    # 用等待时间占比做一个教学用的粗略分类。
    if not 0 <= wait_ratio <= 1:
        raise ValueError("wait_ratio 必须在 0 到 1 之间")
    return "I/O 密集型" if wait_ratio >= 0.5 else "CPU 密集型"


examples = {
    "数据库查询": 0.95,
    "纯 Python 质数计算": 0.02,
    "读取很多小文件": 0.80,
}

for task, ratio in examples.items():
    print(f"{task}: {classify_task(ratio)}")

数据库查询: I/O 密集型
纯 Python 质数计算: CPU 密集型
读取很多小文件: I/O 密集型


## 5. GIL：为什么 Python 线程不一定能加速计算

GIL 的全称是 **Global Interpreter Lock（全局解释器锁）**。它主要是 CPython 解释器的实现机制，并不是 Python 语言规范要求所有 Python 实现都必须具备的特性。

在项目当前使用的标准 CPython 构建中，可以先建立这样的心智模型：

```text
一个 Python 进程
├── 线程 A ─┐
├── 线程 B ─┼──> 同一时刻通常只有一个线程持有 GIL 并执行 Python 字节码
└── 线程 C ─┘
```

线程仍然是真实的操作系统线程，也仍然能够并发工作。关键在于线程此时在做什么：

| 工作类型 | GIL 的典型影响 | 是否适合多线程 |
|---|---|---|
| 等待网络、磁盘、数据库 | 阻塞等待时通常会释放 GIL，让其他线程执行 | 适合 |
| `sleep` 等等待操作 | 等待期间其他线程可以继续 | 适合 |
| 纯 Python 数值循环 | 多个线程会竞争 GIL，通常不能同时执行 Python 字节码 | 通常不适合 |
| NumPy、压缩等原生扩展 | 扩展可能在耗时计算时主动释放 GIL | 取决于扩展实现 |

因此，GIL **不会让线程失去价值**：线程依然很适合 I/O 密集型任务；但纯 Python CPU 密集型任务通常应考虑多进程、能够释放 GIL 的原生库，或其他计算方案。

### GIL 不等于线程安全

GIL 保护的是解释器内部状态，不是你的业务数据。`读取余额 → 计算新余额 → 写回余额` 这类多步操作仍可能被线程切换打断，因此共享可变状态仍需要 `Lock`、线程安全队列或避免共享。

### Python 3.13 的 free-threaded 构建

Python 3.13 提供了可选的 free-threaded（禁用 GIL）构建，但标准构建仍启用 GIL。是否使用 free-threaded 解释器是运行环境属性，第三方扩展也需要具备相应兼容性；不能仅根据“Python 版本是 3.13”就假设没有 GIL。

In [5]:
import platform
import sys


gil_probe = getattr(sys, "_is_gil_enabled", None)
gil_status = gil_probe() if gil_probe is not None else "当前解释器未提供检测接口"

print("Python 实现：", platform.python_implementation())
print("Python 版本：", platform.python_version())
print("GIL 是否启用：", gil_status)

Python 实现： CPython
Python 版本： 3.13.12
GIL 是否启用： True


## 6. 与 Java 的概念对照

| Java 常见概念 | Python 中接近的概念 | 重要差异 |
|---|---|---|
| `Thread` | `threading.Thread` | 都是操作系统线程；CPython 默认构建还要考虑 GIL |
| `ExecutorService` | `concurrent.futures.ThreadPoolExecutor` | 都用线程池提交任务 |
| `Future<T>` | `concurrent.futures.Future` / `asyncio.Future` | 两种 Python Future 属于不同并发体系，不应随意混用 |
| `CompletableFuture` | `asyncio.Task` + `await` | Python 更强调协程语法和事件循环 |
| 虚拟线程 | 没有一一对应物 | `asyncio` 是协作式协程，不是“更轻的 Thread” |

最关键的差异是：`async def` 函数被调用时不会自动创建线程，它会返回一个协程对象，需要被 `await` 或由事件循环调度。

## 7. 初步选择指南

1. 代码简单、任务数量少：优先同步代码。
2. 需要复用阻塞式 I/O 库：使用线程池。
3. 大量连接或任务都支持 `async/await`：使用 `asyncio` 或 AnyIO。
4. 纯 Python CPU 计算：使用多进程，或调用会释放 GIL 的原生计算库。

不要为了“看起来先进”而异步化。异步会引入任务生命周期、取消、超时和错误传播等额外问题。

## 8. 常见误区

- **误区：异步一定比同步快。** 单个短任务通常不会，异步还会增加调度开销。
- **误区：异步等于多线程。** `asyncio` 默认只使用一个线程。
- **误区：并发等于并行。** 单线程事件循环可以并发，但不能并行执行纯 Python 代码。
- **误区：加线程就能加速所有任务。** 默认 CPython 的 GIL 会限制纯 Python CPU 计算的多线程并行。

## 9. 小练习

判断下面任务更适合同步、线程池、异步还是多进程：

1. 依次读取三个很小的配置文件，只在启动时执行一次。
2. 同时请求一千个支持异步客户端的 HTTP 地址。
3. 调用一个只有同步接口的旧数据库驱动，并发量约十个。
4. 用纯 Python 对一百张大图执行复杂像素计算。

建议答案：1 同步；2 异步；3 线程池；4 多进程或释放 GIL 的图像库。

## 本章小结

选择并发方案前，先回答两个问题：任务的大部分时间是在**等待**还是在**计算**？现有库提供的是**同步接口**还是**异步接口**？下一章将使用线程和进程分别解决这两类问题。